# L53 · 🎨 MFCC 图形化 — 波形 → 声谱图（spectrogram） → Mel 谱 → 倒谱系数，逐层图示

**目标**：用图展示 DCT 如何压缩 Mel 频谱，直观理解 MFCC 前几个系数的含义。

🔗 Aurora 连接：`aurora.audio.mel.mel_spectrogram` · `aurora.audio.mfcc.mfcc` · `aurora.audio.mfcc.dct_ii`

## 学习目标

学完本课，你应能：

1. 解释 DCT-II 为何能把平滑的 Mel 频谱包络压缩到前几个系数（能量集中原理）。
2. 说出 c[0]、c[1]、c[2] 各自编码什么物理量（总能量、倾斜、弯曲）。
3. 调用 `aurora.audio.mfcc.mfcc()` 生成完整的 MFCC 矩阵并以热力图展示。
4. 用 `power_to_db` 替代手写 `log10`，与 `mfcc.py` 保持实现一致。

## 核心直觉

Mel 频谱有 40–80 个 bin，相邻 bin 之间高度相关（频谱包络（spectral envelope）变化平滑）。
DCT-II 把这种"慢变化"集中到前几个系数，后面的系数几乎是噪声——这就是 MFCC 能用 13 维描述音色的原因。
倒谱（cepstrum）= 对数谱的逆傅里叶变换（IDFT），把卷积（声道 × 激励）拆成加法，让每维系数承载独立的物理含义。MFCC 用 DCT-II 近似 IDFT，计算更高效。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from aurora.audio.io import chirp
from aurora.audio.mel import mel_spectrogram, power_to_db
from aurora.audio.mfcc import mfcc, dct_ii

## 1. DCT 压缩：能量集中原理

Mel 频谱每帧是一个 40 维向量 `S[m]`（m = 0…39），经 DCT-II 得到系数：

```
y[k] = sum_{n=0}^{N-1}  S[n] * cos( pi/N * (n + 0.5) * k )
```

正交归一化（`norm="ortho"`）后，`k=0` 对应常数基函数，`k=1` 对应最慢余弦……
频谱包络平滑 → 高阶系数（大 k）贡献极小 → 截取前 13 个系数损失信息极少。

In [ ]:
SR = 22050
N_MELS = 40

# 合成一段混合音：基频 440 Hz + 泛音，模拟元音音色
t = np.linspace(0, 1.0, SR, endpoint=False)
signal = (
    0.6 * np.sin(2 * np.pi * 440 * t)
    + 0.3 * np.sin(2 * np.pi * 880 * t)
    + 0.1 * np.sin(2 * np.pi * 1320 * t)
)

# 取单帧 Mel 频谱（取第 10 帧，跳过静音起始）
mel_spec = mel_spectrogram(signal, SR, n_fft=1024, n_mels=N_MELS)
log_mel_frame = power_to_db(mel_spec[10], top_db=None)   # 单帧 log-mel，形状 (40,)  — 与 mfcc.py 一致

# DCT 变换
dct_all = dct_ii(log_mel_frame, norm="ortho")          # 形状 (40,)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].bar(range(N_MELS), log_mel_frame, color="#4C72B0", width=0.8)
axes[0].set_title("Log-Mel 频谱（单帧，40 维）", fontsize=12)
axes[0].set_xlabel("Mel bin 索引")
axes[0].set_ylabel("dB")

axes[1].bar(range(N_MELS), dct_all, color="#DD8452", width=0.8)
axes[1].axvline(12.5, color="red", linestyle="--", linewidth=1.5, label="截止 k=13")
axes[1].set_title("DCT-II 输出（40 个系数）", fontsize=12)
axes[1].set_xlabel("DCT 系数索引 k")
axes[1].set_ylabel("系数值")
axes[1].legend()

plt.tight_layout()
plt.savefig("v1_dct_compression.png", dpi=120)
plt.show()
print("✅ 观察：能量集中在 k < 13 的左侧，右侧系数接近 0")

## 2. 各阶系数的物理含义

| 系数 | 基函数形状 | 物理含义 |
|------|-----------|---------|
| `c[0]` | 常数（全1） | log-mel 频谱的均值 = 帧总对数能量 |
| `c[1]` | 半周期余弦 | 频谱包络的低频倾斜（高频能量 vs 低频能量） |
| `c[2]` | 一周期余弦 | 包络的"弓形"弯曲 |
| `c[3..12]` | 更高频余弦 | 越来越细致的共振峰（formant）结构 |
| `c[13+]` | 高频抖动 | 噪声，通常丢弃 |

关键公式（`norm="ortho"`，`k=0`）：

```
c[0] = sqrt(1/N) * sum_{n} S[n]   # 即 N 个 Mel bin 的均值（乘以 sqrt(N)）
```

c[0] 随音量变化，做说话人无关识别时常将其去掉或用 CMN（倒谱均值归一化）消除。

In [ ]:
# 对比三种音色：纯正弦 vs 共振峰聚集在高频 vs chirp
t = np.linspace(0, 1.0, SR, endpoint=False)

sig_low  = np.sin(2 * np.pi * 300 * t)                          # 纯低频
sig_high = np.sin(2 * np.pi * 3000 * t)                         # 纯高频
sig_mix  = 0.5 * sig_low + 0.5 * sig_high                       # 低+高

signals  = [sig_low, sig_high, sig_mix]
labels   = ["纯低频 300 Hz", "纯高频 3000 Hz", "低+高 混合"]
colors   = ["#4C72B0", "#DD8452", "#55A868"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)

for ax, sig, label, color in zip(axes, signals, labels, colors):
    mel_s = mel_spectrogram(sig, SR, n_fft=1024, n_mels=N_MELS)
    lm = power_to_db(mel_s[10], top_db=None)
    c = dct_ii(lm, norm="ortho")[:13]
    ax.bar(range(13), c, color=color, width=0.8)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("MFCC 系数 k")
    if ax is axes[0]:
        ax.set_ylabel("系数值")
    # 标注 c0
    ax.annotate(f"c[0]={c[0]:.1f}", xy=(0, c[0]), xytext=(2, c[0]+1),
                arrowprops=dict(arrowstyle="->", color="gray"), fontsize=9)

fig.suptitle("不同音色的前 13 个 MFCC 系数对比", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("v1_mfcc_timbre.png", dpi=120)
plt.show()
print("✅ 观察：c[0] 反映总能量；c[1] 正 → 低频主导，c[1] 负 → 高频主导")

## 参数实验：能量集中度量化

**参数**：`N_MELS = 40`，截取 `n_mfcc = 13`

**预期现象**：
- 前 13 个系数的方差之和 >> 后 27 个系数的方差之和
- 能量保留率（前13维 / 全40维的功率比）应 > 95%

调整 `N_MELS` 到 80 或改变信号复杂度（chirp vs 纯音），观察能量保留率如何变化。

In [ ]:
SR = 22050
N_MELS = 40
N_KEEP = 13

# 用一段 chirp 信号（频谱更丰富，更难压缩）
t = np.linspace(0, 1.0, SR, endpoint=False)
sig_chirp = chirp(duration=1.0, f0=200, f1=4000, sample_rate=SR)

mel_s = mel_spectrogram(sig_chirp, SR, n_fft=1024, n_mels=N_MELS)

# 对所有帧做 DCT，计算每个 bin 的方差
log_mel_all = power_to_db(mel_s, top_db=None)   # (n_frames, N_MELS) — 与 mfcc.py 一致
dct_all_frames = dct_ii(log_mel_all, norm="ortho")  # (n_frames, N_MELS)

var_per_bin = np.var(dct_all_frames, axis=0)  # (N_MELS,)
power_total  = var_per_bin.sum()
power_kept   = var_per_bin[:N_KEEP].sum()
retention    = power_kept / power_total * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

# 各系数方差
axes[0].bar(range(N_MELS), var_per_bin, color="#4C72B0", width=0.8)
axes[0].axvline(N_KEEP - 0.5, color="red", linestyle="--", lw=1.5, label=f"截止 k={N_KEEP}")
axes[0].set_title("DCT 系数方差（跨帧统计）", fontsize=12)
axes[0].set_xlabel("DCT 系数索引 k")
axes[0].set_ylabel("方差")
axes[0].legend()

# 累积能量保留率
cumulative = np.cumsum(var_per_bin) / power_total * 100
axes[1].plot(range(N_MELS), cumulative, marker="o", markersize=4, color="#DD8452")
axes[1].axhline(95, color="gray", linestyle=":", lw=1.5, label="95% 阈值")
axes[1].axvline(N_KEEP - 1, color="red", linestyle="--", lw=1.5, label=f"k={N_KEEP-1}")
axes[1].set_title("累积方差保留率（%）", fontsize=12)
axes[1].set_xlabel("截取前 k 个系数")
axes[1].set_ylabel("保留率 (%)")
axes[1].legend()

plt.tight_layout()
plt.savefig("v1_energy_retention.png", dpi=120)
plt.show()

print(f"✅ N_MELS={N_MELS}，保留前 {N_KEEP} 维：")
print(f"   前 {N_KEEP} 系数方差和 = {power_kept:.4f}")
print(f"   后 {N_MELS-N_KEEP} 系数方差和 = {power_total - power_kept:.4f}")
print(f"   能量保留率 = {retention:.1f}%")
assert retention > 80.0, f"能量保留率 {retention:.1f}% 低于预期（应 > 80%）"


In [ ]:
# MFCC 全链路演示：mfcc() 热力图
# 完整链路：mel_spectrogram -> power_to_db -> dct_ii -> [:, :n_mfcc]
SR_demo = 22050
N_MFCC = 13

# chirp 信号：频率随时间变化，MFCC 热力图更有代表性
sig_demo = chirp(duration=2.0, f0=200, f1=4000, sample_rate=SR_demo)

# 调用 aurora 完整 mfcc() 函数
mfcc_matrix = mfcc(sig_demo, SR_demo, n_mfcc=N_MFCC, n_fft=1024, n_mels=40)  # (n_frames, 13)

print(f'MFCC 矩阵形状：{mfcc_matrix.shape}  -> (n_frames, n_mfcc)')

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(
    mfcc_matrix.T,           # 转置：y 轴 = 系数编号，x 轴 = 时间帧
    aspect='auto',
    origin='lower',
    interpolation='nearest',
    cmap='RdBu_r'
)
ax.set_title('MFCC 热力图（chirp 信号，全链路 mfcc() 输出）', fontsize=13)
ax.set_xlabel('时间帧')
ax.set_ylabel('MFCC 系数编号')
ax.set_yticks(range(N_MFCC))
ax.set_yticklabels([f'c[{k}]' for k in range(N_MFCC)], fontsize=8)
fig.colorbar(im, ax=ax, label='系数值 (dB 域)')
plt.tight_layout()
plt.show()
print('✅ 完整 MFCC 链路验证：mfcc() 端到端输出已可视化')

In [ ]:
# ── TODO 练习：DCT 重构误差 ────────────────────────────────────────────
# 目标：用前 N_KEEP 个 MFCC 系数重构 log-Mel 频谱，量化信息损失。
#
# DCT-II 的正交性：将 dct_truncated 乘以 DCT-II 基矩阵的转置可近似还原原始向量。
# 步骤：
#   1. 取 dct_all 的前 N_KEEP 列，其余置零，得到 dct_truncated
#   2. 构建 DCT-II 正交基矩阵 B，形状 (N_MELS, N_MELS)
#      B[k, n] = cos(pi/N * (n+0.5) * k)，norm='ortho' 缩放
#   3. 逆变换：log_mel_recon = dct_truncated @ B （B 列正交 -> 近似逆变换）
#   4. 计算 RMS 误差并断言 < 5 dB

SR = 22050
N_MELS = 40
N_KEEP = 13

sig_todo = chirp(duration=1.0, f0=200, f1=4000, sample_rate=SR)
mel_todo = mel_spectrogram(sig_todo, SR, n_fft=1024, n_mels=N_MELS)
log_mel_todo = power_to_db(mel_todo, top_db=None)   # (n_frames, N_MELS)
dct_todo = dct_ii(log_mel_todo, norm='ortho')        # (n_frames, N_MELS)

# === 请在下方实现 ===
# 1. 截断系数
dct_truncated = dct_todo.copy()
dct_truncated[:, N_KEEP:] = 0.0

# 2. TODO: 构建 DCT-II 正交基矩阵 B 并做逆变换
raise NotImplementedError(
    'TODO: 构建 B[k,n] = cos(pi/N*(n+0.5)*k) 基矩阵（ortho 缩放）'
    '，然后 log_mel_recon = dct_truncated @ B，'
    '再计算 RMS 误差后注释掉本行。'
)

# 验证（完成实现后取消注释）：
# rms_error = np.sqrt(np.mean((log_mel_recon - log_mel_todo) ** 2))
# print(f'重构 RMS 误差：{rms_error:.2f} dB')
# assert rms_error < 5.0, f'误差 {rms_error:.2f} dB 超出预期（应 < 5 dB）'
# print('✅ 截取 13 维重构误差在可接受范围内')

## 本课收束

本节用 `dct_ii` 将 40 维 log-Mel 频谱压缩，展示了 DCT 系数的方差如何以指数速度衰减，前 13 个系数可保留 95%+ 的方差信息。
`mfcc` 函数输出形状为 `(n_frames, n_mfcc)` 的矩阵，每行是一帧音频的 MFCC 向量，c[0] 编码总对数能量，c[1]/c[2] 编码频谱包络的粗粒度倾斜与弯曲。
这两个函数均属于 `aurora.audio.mfcc` 模块，从 log-Mel 到系数的完整链路为：`mel_spectrogram → power_to_db → dct_ii → 截取前 n_mfcc 列`。
下节（**L54**）将开启深度学习模块，从零实现自动微分引擎（Value 节点与计算图），为神经网络训练打好基础。